In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
load_dotenv(override=True)

In [ ]:
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

In [ ]:
answers = []
evaluations = []
task = "Please write a 20 word explanation for why most humans like chocolate. "

openai = OpenAI()
llm_generator = "gpt-5-nano"

gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
llm_evaluator = "gemini-2.5-flash"

In [ ]:
count = 0
approved = False
feedback = ""
while not approved and count < 3:
    messages = [{"role": "user", "content": task + ". Take into account this feedback:" + feedback}]
    response = openai.chat.completions.create(model=llm_generator, messages=messages)
    answer = response.choices[0].message.content    
    answers.append(answer)    

    message_for_evaluator = f"""You are an LLM evaluator, evaluate this explanation: \"{answer}\" 
    That explanation was given by another LLM to this task: "{task}".     
    If you don't agree with the explanation respond with 5 word feedback.
    The explanation should mention something about neurotransmitters, aroma, comfort and color.
    If you aggree with the answer respond with \"approved\".""" 
    messages = [{"role": "user", "content": message_for_evaluator}]
    response = gemini.chat.completions.create(model=llm_evaluator, messages=messages)
    feedback = response.choices[0].message.content      
    evaluations.append(feedback)

    if feedback == "approved":        
        approved = True        
    count += 1

In [ ]:
for answer, evaluation in zip(answers, evaluations):
    print(f"Answer: {answer}\nEvaluation: {evaluation}\n")